# [재학습 · 2026-10-08] β=30 기준 평가 + DDQN 라우터

재학습한 모델(`models/rt1006/`)로 학습 없이 평가만 다시 하고, 마지막에 DDQN 라우터만 새로 학습한다(GitHub 커밋 `0e86a41`의 코드, 배포 β만 −30 → 30으로 치환).
1. §23 v4 다양성 진단(4.4절) — ρ 분포, β별 평균 ρ, top-10 순환 종목 수
2. §24 노출 맞춤 벤치마크(6.11절)
3. §18 거래비용 스윕(6.9절) — 5/10/15/20bp
4. §25 저턴오버 아티팩트 진단(4.4절)
5. §19 블록 순열검정(6.10절)
6. §22 DDQN 매칭 라우터(8장 한계#3) — DQN 매칭 라우터는 연쇄 재학습에서 만든 것을 불러오고, DDQN만 20k 스텝 학습

- **`런타임 > 모두 실행`만 누르면 된다.** 끊기면 다시 `모두 실행` — DDQN 학습은 캐시에서 불러온다. CPU로 충분, 약 1~2시간.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과 CSV는 원본과 같은 이름으로 `data/crypto/rt1006/`에 저장된다(DDQN 쪽만 `_fix1004` 꼬리를 뺀 이름).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·v4 풀·원본 라우터 로드 (학습 없음)
run_cells('crypto/notebooks/rerun_fix1004/crypto_eval2_rerun_fix1004.ipynb', [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], patch_after=2)


In [ ]:
# 1. §23 다양성 진단 / 2. §24 노출 맞춤 벤치마크 (β=30)
run_cells('crypto/notebooks/rerun_fix1004/crypto_eval2_rerun_fix1004.ipynb', [14, 16], patch_after=None, subs={16: [('ALPHA_DEPLOY, BETA_DEPLOY = 0.01, -30', 'ALPHA_DEPLOY, BETA_DEPLOY = 0.01, 30')], 21: [("(' (배포)' if beta == -30 else '')", "(' (배포)' if beta == 30 else '')")]})


In [ ]:
# 3. §18 거래비용 스윕 (β=30, α=0.01)
run_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb', [94], patch_after=None, subs={94: [('(-30, ALPHA_DEPLOY', '(30, ALPHA_DEPLOY'), ('β=-30, α=0.01, valid/test', 'β=30, α=0.01, valid/test')]})


In [ ]:
# 4. §25 저턴오버 진단 / 5. §19 블록 순열검정
run_cells('crypto/notebooks/rerun_fix1004/crypto_eval2_rerun_fix1004.ipynb', [19, 21], patch_after=None, subs={16: [('ALPHA_DEPLOY, BETA_DEPLOY = 0.01, -30', 'ALPHA_DEPLOY, BETA_DEPLOY = 0.01, 30')], 21: [("(' (배포)' if beta == -30 else '')", "(' (배포)' if beta == 30 else '')")]})


In [ ]:
# 6. §22 DDQN 매칭 라우터 — valid 구간·풀 로드 → 매칭 라우터 로드 → DDQN 학습 → 비교
run_cells('crypto/notebooks/rerun_fix1004/crypto_ddqn_retrain_fix1004.ipynb', [9], patch_after=None)
_ip.run_cell("# §13-I-A 매칭 라우터 — 연쇄 재학습(crypto_cascade_retrain_rt1006)에서 학습한 것을 그대로 불러온다\nrouter_v4_matched = DQN.load(str(MODEL_DIR / 'crypto_hrl_router_v4_matched.zip'))\nprint('[v4] 매칭 라우터(DQN) 로드 완료')\n")
run_cells('crypto/notebooks/rerun_fix1004/crypto_ddqn_retrain_fix1004.ipynb', [11, 12, 13], patch_after=None, subs={12: [("'crypto_hrl_router_v4_ddqn_fix1004.zip'", "'crypto_hrl_router_v4_ddqn.zip'")], 13: [("'crypto_hrl_v4_dqn_ddqn_router_fix1004.csv'", "'crypto_hrl_v4_dqn_ddqn_router.csv'")]})
print('\n전부 완료')
